# Argumind — Fine-Tuning IndoBERT (F-01) & Cross-Encoder NLI (F-02) di Google Colab

Notebook ini menjalankan pelatihan terstruktur sesuai **`research/METHODOLOGY.md`**, **`research/EXPERIMENT_PLAN.md`**, dan **`research/REPRODUCIBILITY.md`**:
1. Menghubungkan Google Colab ke repositori GitHub **`salsabilaputri95/argumind-project`** (mendukung repositori **Private** dengan `GITHUB_TOKEN` maupun **Public**).
2. Menginstal dependensi Python yang kompatibel dengan Python 3.12/3.13 Colab & memverifikasi integritas checksum `manifest.sha256`.
3. Melatih model **IndoBERT BIO Sequence Labeler** (`F-01`: `Claim`, `ExplicitPremise`, `Evidence`).
4. Melatih **Cross-Encoder NLI Relation Classifier** (`F-02`: `Support`, `Attack`, `NoRelation`) + dekode **Chu-Liu/Edmonds DAG** (`cycle_rate == 0.0`).
5. Mengekspor metrik mentah (`eval_bio.json`, `eval_nli.json`, `metrics_raw.json`) dan bobot model ke Google Drive atau unduhan lokal.

In [ ]:
# @title 1. Cek GPU Google Colab & Kloning Repositori GitHub (Mendukung Repo Private & Public)
!nvidia-smi

import os
from pathlib import Path

REPO_OWNER_AND_NAME = "salsabilaputri95/argumind-project"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
GITHUB_TOKEN = ""  # @param {type:"string"}
WORK_DIR = Path("/content/argumind-project")

# Ambil GITHUB_TOKEN dari Colab Secrets (ikon kunci di panel kiri) jika kolom di atas kosong
if not GITHUB_TOKEN.strip():
    try:
        from google.colab import userdata

        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN") or ""
    except Exception:
        GITHUB_TOKEN = ""

if GITHUB_TOKEN.strip():
    CLONE_URL = f"https://{GITHUB_TOKEN.strip()}@github.com/{REPO_OWNER_AND_NAME}.git"
else:
    CLONE_URL = f"https://github.com/{REPO_OWNER_AND_NAME}.git"

if (WORK_DIR / ".git").exists():
    %cd /content/argumind-project
    !git remote set-url origin {CLONE_URL}
    !git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {CLONE_URL} /content/argumind-project

if not (WORK_DIR / ".git").exists():
    raise RuntimeError(
        "Gagal mengkloning repositori! Karena repositori GitHub Anda masih berstatus PRIVATE, "
        "pilih salah satu cara berikut:\n"
        "1. Masukkan GitHub Personal Access Token (PAT) pada kolom GITHUB_TOKEN di Sel 1 "
        "(atau tambahkan secret 'GITHUB_TOKEN' di menu Secrets 🔑 panel kiri Colab), ATAU\n"
        "2. Ubah visibilitas repo GitHub 'salsabilaputri95/argumind-project' menjadi Public sementara saat pelatihan di Colab."
    )

os.chdir(WORK_DIR)
!pwd
!git log -n 1 --oneline

In [ ]:
# @title 2. Instal Dependensi Python (Kompatibel Python 3.12/3.13 Colab) & Atur PYTHONPATH
!pip install -q "transformers>=4.46.0" "datasets>=3.1.0" "accelerate>=1.1.0" \
  "networkx>=3.4.0" "pydantic>=2.10.0" "bert-score>=0.3.13" "krippendorff>=0.8.0" \
  python-docx pymupdf

import sys

import torch

if not WORK_DIR.exists():
    raise RuntimeError("Direktori /content/argumind-project belum ada! Pastikan Sel 1 berhasil dijalankan.")

os.chdir(WORK_DIR)

paths_to_add = []
for rel in [
    "argumind-platform/packages/shared-schemas/src",
    "argumind-platform/packages/evaluation-metrics/src",
    "argumind-platform/apps/nlp-worker/src",
    "argumind-platform/apps/api/src",
    ".",
]:
    p = str((WORK_DIR / rel).resolve())
    paths_to_add.append(p)
    if p not in sys.path:
        sys.path.insert(0, p)

os.environ["PYTHONPATH"] = ":".join(paths_to_add)
print("Working Directory:", os.getcwd())
print("PYTHONPATH:", os.environ["PYTHONPATH"])
print("PyTorch:", torch.__version__, "| CUDA Available:", torch.cuda.is_available())

In [ ]:
# @title 3. Verifikasi Checksum Dataset (manifest.sha256)
os.chdir(WORK_DIR)
DATA_DIR = "research/data/processed/v1.0"  # @param {type:"string"}
!cd {DATA_DIR} && sha256sum -c manifest.sha256
!wc -l {DATA_DIR}/*.jsonl

In [ ]:
# @title 4. Fine-Tuning IndoBERT BIO Component Extractor (F-01)
os.chdir(WORK_DIR)
MODEL_NAME = "indobenchmark/indobert-base-p2"  # @param ["indobenchmark/indobert-base-p2", "indobenchmark/indobert-large-p2"]
SEED = 42  # @param [42, 100, 2026] {type:"raw"}
EPOCHS = 10  # @param {type:"integer"}
BATCH_SIZE = 16  # @param {type:"integer"}
LR = 2e-5  # @param {type:"number"}

!PYTHONPATH=$PYTHONPATH python3 -m research.runners.train_indobert_bio \
  --data-dir {DATA_DIR} \
  --output-dir research/outputs/colab_indobert_bio_seed{SEED} \
  --model-name {MODEL_NAME} \
  --seed {SEED} \
  --epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --lr {LR}

In [ ]:
# @title 5. Fine-Tuning Cross-Encoder NLI Relation Classifier + DAG Decoder (F-02)
os.chdir(WORK_DIR)
NLI_BASE_MODEL = "indobenchmark/indobert-base-p2"  # @param ["indobenchmark/indobert-base-p2", "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"]
NLI_EPOCHS = 8  # @param {type:"integer"}

!PYTHONPATH=$PYTHONPATH python3 -m research.runners.train_nli_relations \
  --data-dir {DATA_DIR} \
  --output-dir research/outputs/colab_nli_rel_seed{SEED} \
  --model-name {NLI_BASE_MODEL} \
  --seed {SEED} \
  --epochs {NLI_EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --lr {LR}

In [ ]:
# @title 6. Jalankan Runner Evaluasi Standar E-01 & E-03b serta Tampilkan Hasil
import json

os.chdir(WORK_DIR)
!PYTHONPATH=$PYTHONPATH python3 -m research.runners.run_experiment \
  --exp-id E-01 --arch M1_ArgumindHybrid \
  --data-dir {DATA_DIR} --split test --seed {SEED} \
  --bio-model-dir research/outputs/colab_indobert_bio_seed{SEED}/best_model \
  --nli-model-dir research/outputs/colab_nli_rel_seed{SEED}/best_nli_model \
  --output-dir research/outputs/E-01/colab_seed{SEED}

!PYTHONPATH=$PYTHONPATH python3 -m research.runners.run_experiment \
  --exp-id E-03b --arch M3_GraphGroundedScoring \
  --data-dir {DATA_DIR} --split test --seed {SEED} \
  --output-dir research/outputs/E-03b/colab_seed{SEED}

for path in [
    f"research/outputs/colab_indobert_bio_seed{SEED}/eval_bio.json",
    f"research/outputs/colab_nli_rel_seed{SEED}/eval_nli.json",
    f"research/outputs/E-01/colab_seed{SEED}/metrics_raw.json",
    f"research/outputs/E-03b/colab_seed{SEED}/metrics_raw.json",
]:
    p = WORK_DIR / path
    if p.exists():
        print(f"=== {path} ===")
        print(json.dumps(json.loads(p.read_text()), indent=2))
    else:
        print(f"[WARNING] File belum ditemukan: {path}")

In [ ]:
# @title 7. Simpan Checkpoint & Artefak ke Google Drive / Unduh ZIP
os.chdir(WORK_DIR)
!zip -r -q argumind_colab_outputs_seed{SEED}.zip research/outputs/
zip_path = WORK_DIR / f"argumind_colab_outputs_seed{SEED}.zip"
print("Arsip siap:", zip_path, "| Exists:", zip_path.exists())

SAVE_TO_DRIVE = False  # @param {type:"boolean"}
if SAVE_TO_DRIVE and zip_path.exists():
    from google.colab import drive

    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/argumind_outputs
    !cp {zip_path} /content/drive/MyDrive/argumind_outputs/
    print("Tersalin ke Google Drive!")